In [ ]:
from datasets import load_dataset
dataset=load_dataset("tuetschek/atis")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'intent', 'text', 'slots'],
        num_rows: 4978
    })
    test: Dataset({
        features: ['id', 'intent', 'text', 'slots'],
        num_rows: 893
    })
})

In [ ]:
dataset['train'][0]

{'id': 0,
 'intent': 'flight',
 'text': 'i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'slots': 'O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day'}

In [ ]:
def preprocess(example):
    return {'tokens':example['text'].split(),
            'slot_labels':example['slots'].split()}

In [ ]:
dataset=dataset.map(preprocess)

Map: 100%|██████████| 893/893 [00:00<00:00, 18694.38 examples/s]


In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'intent', 'text', 'slots', 'tokens', 'slot_labels'],
        num_rows: 4978
    })
    test: Dataset({
        features: ['id', 'intent', 'text', 'slots', 'tokens', 'slot_labels'],
        num_rows: 893
    })
})

In [ ]:
dataset['train'][0]

{'id': 0,
 'intent': 'flight',
 'text': 'i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'slots': 'O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day',
 'tokens': ['i',
  'want',
  'to',
  'fly',
  'from',
  'boston',
  'at',
  '838',
  'am',
  'and',
  'arrive',
  'in',
  'denver',
  'at',
  '1110',
  'in',
  'the',
  'morning'],
 'slot_labels': ['O',
  'O',
  'O',
  'O',
  'O',
  'B-fromloc.city_name',
  'O',
  'B-depart_time.time',
  'I-depart_time.time',
  'O',
  'O',
  'O',
  'B-toloc.city_name',
  'O',
  'B-arrive_time.time',
  'O',
  'O',
  'B-arrive_time.period_of_day']}

In [ ]:
from collections import Counter
counter=Counter()
for example in dataset['train']:
    counter.update(token for token in example['tokens'])
vocab={'PAD':0,'UNK':1}
for word in counter:
    vocab[word]=len(vocab)

In [ ]:
len(vocab)

891

In [ ]:
slotset=set()
for example in dataset['train']:
    slotset.update(slot for slot in example['slot_labels'])

In [ ]:
slot2index={slot:index for index,slot in enumerate(slotset)}
index2slot={index:slot for slot,index in slot2index.items()}

In [ ]:
intentset=set(example['intent'] for example in dataset['train'])

In [ ]:
intent2index={intent:index for index,intent in enumerate(intentset)}
index2intent={index:intent for intent,index in intent2index.items()}

In [ ]:
def encode(example):
    tokens=example['tokens']
    slot_labels=example['slot_labels']
    intent=example['intent']
    return {'input_ids':[vocab.get(token,1) for token in tokens],
    'slot_ids':[slot2index.get(slot,0) for slot in slot_labels],
    'intent_id':intent2index.get(intent,0)}

In [ ]:
dataset=dataset.map(encode)

Map: 100%|██████████| 893/893 [00:00<00:00, 10575.92 examples/s]


In [ ]:
dataset['train'][0]

{'id': 0,
 'intent': 'flight',
 'text': 'i want to fly from boston at 838 am and arrive in denver at 1110 in the morning',
 'slots': 'O O O O O B-fromloc.city_name O B-depart_time.time I-depart_time.time O O O B-toloc.city_name O B-arrive_time.time O O B-arrive_time.period_of_day',
 'tokens': ['i',
  'want',
  'to',
  'fly',
  'from',
  'boston',
  'at',
  '838',
  'am',
  'and',
  'arrive',
  'in',
  'denver',
  'at',
  '1110',
  'in',
  'the',
  'morning'],
 'slot_labels': ['O',
  'O',
  'O',
  'O',
  'O',
  'B-fromloc.city_name',
  'O',
  'B-depart_time.time',
  'I-depart_time.time',
  'O',
  'O',
  'O',
  'B-toloc.city_name',
  'O',
  'B-arrive_time.time',
  'O',
  'O',
  'B-arrive_time.period_of_day'],
 'input_ids': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 8, 15, 13, 16, 17],
 'slot_ids': [78,
  78,
  78,
  78,
  78,
  38,
  78,
  55,
  32,
  78,
  78,
  78,
  94,
  78,
  100,
  78,
  78,
  61],
 'intent_id': 7}

In [ ]:
import torch
def padding(batch):
    input_ids=[]
    slot_ids=[]
    mask=[]
    intent_id=[]
    maxlen=max(len(example['input_ids']) for example in batch)
    for example in batch:
        L=len(example['input_ids'])
        pad_len=maxlen-L
        input_ids.append(example['input_ids']+[0]*pad_len)
        slot_ids.append(example['slot_ids']+[0]*pad_len)
        mask.append([1]*L+[0]*pad_len)
        intent_id.append(example['intent_id'])
    return{'input_ids':torch.tensor(input_ids,dtype=torch.long),
               'slot_ids':torch.tensor(slot_ids,dtype=torch.long),
               'mask':torch.tensor(mask,dtype=torch.float),
              'intent_id':torch.tensor(intent_id,dtype=torch.long)}

In [ ]:
from torch.utils.data import DataLoader
train_loader=DataLoader(dataset['train'],batch_size=32,shuffle=True,collate_fn=padding)
test_loader=DataLoader(dataset['test'],batch_size=32,shuffle=True,collate_fn=padding)

In [ ]:
train_loader

In [ ]:
import torch
import torch.nn as nn
class LSTMScratchCell(nn.Module):
    def __init__(self,embedding_dim,hidden_dim):
        super(LSTMScratchCell,self).__init__()
        self.input_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.output_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.forget_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
        self.candidate_layer=nn.Linear(embedding_dim+hidden_dim,hidden_dim)
    def forward(self,inp,h,c):
        concatenated=torch.cat((inp,h),dim=1)
        i=torch.sigmoid(self.input_layer(concatenated))
        o=torch.sigmoid(self.output_layer(concatenated))
        f=torch.sigmoid(self.forget_layer(concatenated))
        cand=torch.tanh(self.candidate_layer(concatenated))
        c=i*cand+f*c
        h=o*torch.tanh(c)
        return h,c



In [ ]:
class BiLSTMCell(nn.Module):
    def __init__(self,embedding_dim,hidden_dim):
        super(BiLSTMCell,self).__init__()
        self.hidden_dim=hidden_dim
        self.fwd_cell=LSTMScratchCell(embedding_dim,hidden_dim)
        self.bwd_cell=LSTMScratchCell(embedding_dim,hidden_dim)
    def forward(self,inp,mask):
        B,T,d=inp.shape
        hf=torch.zeros(B,self.hidden_dim,device=inp.device)
        cf=torch.zeros(B,self.hidden_dim,device=inp.device)

        hb=torch.zeros(B,self.hidden_dim,device=inp.device)
        cb=torch.zeros(B,self.hidden_dim,device=inp.device)

        fwd_outputs=[]
        bwd_outputs=[]
        for t in range(T):
            hf,cf=self.fwd_cell(inp[:,t,:],hf,cf)
            hf=hf*mask[:,t].unsqueeze(1)
            cf=cf*mask[:,t].unsqueeze(1)
            fwd_outputs.append(hf)
        for t in reversed(range(T)):
            hb,cb=self.bwd_cell(inp[:,t,:],hb,cb)
            hb=hb*mask[:,t].unsqueeze(1)
            cb=cb*mask[:,t].unsqueeze(1)
            bwd_outputs.append(hb)
        bwd_outputs.reverse()
        fwd_outputs=torch.stack(fwd_outputs,dim=1)
        bwd_outputs=torch.stack(bwd_outputs,dim=1)
        outputs=torch.cat((fwd_outputs,bwd_outputs),dim=2)
        return outputs,(hf,hb)



In [ ]:
class JointModel(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_slot,num_intent):
        super(JointModel,self).__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim)
        self.bilstm=BiLSTMCell(embedding_dim,hidden_dim)
        self.fc_slot=nn.Linear(2*hidden_dim,num_slot)
        self.fc_intent=nn.Linear(2*hidden_dim,num_intent)
    def forward(self,inp,mask):
        emb=self.embedding(inp)
        lstm_output,(hf,hb)=self.bilstm(emb,mask)
        slot_output=self.fc_slot(lstm_output)
        intent_input=torch.cat((hf,hb),dim=1)
        intent_output=self.fc_intent(intent_input)
        return slot_output,intent_output

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [ ]:
model=JointModel(len(vocab),128,256,len(slot2index),len(intent2index)).to(device)

In [ ]:
def masked_joint_loss(slot_outputs,intent_outputs,slot_true,intent_true,mask):
    B,T,c=slot_outputs.shape
    slot_outputs=slot_outputs.view(B*T,c)
    slot_true=slot_true.view(B*T)
    mask=mask.view(B*T)
    slot_loss=torch.nn.functional.cross_entropy(slot_outputs,slot_true,reduction='none')
    slot_loss=slot_loss*mask
    slot_loss=slot_loss.sum()/mask.sum()
    intent_loss=torch.nn.functional.cross_entropy(intent_outputs,intent_true)
    return slot_loss+intent_loss

In [ ]:
def masked_joint_accuracy(slot_outputs, intent_outputs, slot_true, intent_true, mask):

    # SLOT ACCURACY
    slot_pred = torch.argmax(slot_outputs, dim=-1)

    correct = (slot_pred == slot_true).float()
    correct = correct * mask

    slot_accuracy = correct.sum() / mask.sum()

    # INTENT ACCURACY
    intent_pred = torch.argmax(intent_outputs, dim=-1)

    correct1 = (intent_pred == intent_true).float()
    intent_accuracy = correct1.mean()

    # COMBINED
    return (slot_accuracy + intent_accuracy) / 2

In [ ]:
optimizer=torch.optim.Adam(model.parameters())
from tqdm import tqdm
epochs=10
for epoch in range(epochs):
    total_loss=0
    total_accuracy=0
    model.train()
    loop=tqdm(train_loader,desc=f'Epoch: {epoch+1}/{epochs}')
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        slot_ids = batch['slot_ids'].to(device)
        mask = batch['mask'].to(device)
        intent_ids = batch['intent_id'].to(device)
        #intent_ids=intent_ids.to(device)
        slot_outputs,intent_outputs=model(input_ids,mask)
        batch_loss=masked_joint_loss(slot_outputs, intent_outputs,slot_ids, intent_ids, mask)
        batch_loss.backward()
        optimizer.step()
        total_loss+=batch_loss.item()
        total_accuracy+=masked_joint_accuracy(slot_outputs, intent_outputs,slot_ids, intent_ids, mask).item()
        loop.set_postfix(loss=batch_loss.item())
    print(f'Epoch:{epoch+1} | Loss:{total_loss/len(train_loader)}| Accuracy:{total_accuracy/len(train_loader)}')



Epoch: 1/10:  19%|█▉        | 30/156 [00:04<00:15,  7.90it/s, loss=4.04]